# Exploring Ibis with iacs Data

Ibis provides a backend-agnostic dataframe API that compiles to SQL.
Here we use the DuckDB backend to explore iacs component data.

Key idea: Ibis expressions are **lazy** — nothing executes until you
call `.to_pandas()` or `.execute()`.

In [ ]:
import ibis
import pandas as pd

from iacs.io_system import IOSystem
from iacs.registry import Registry

ibis.options.interactive = True  # auto-display results (like pandas)

## 1. Load iacs data into a Registry

In [ ]:
io = IOSystem()
entity_centered = io.read_entity_centered_file("../components/components.yaml")
registry = Registry.from_entity_centered(entity_centered)

print(f"Component types: {registry.component_types}")

## 2. Connect to DuckDB via Ibis and register tables

In [ ]:
con = ibis.duckdb.connect()

# Register each component table
tables = {}
for comp_type in registry.component_types:
    df = registry.view(comp_type).reset_index()
    table_name = comp_type.replace(" ", "_")
    try:
        con.create_table(table_name, df)
        tables[table_name] = con.table(table_name)
        print(f"Registered '{table_name}' ({len(df)} rows)")
    except Exception as e:
        print(f"Error registering '{table_name}': {e}")

## 3. Basic operations: select, filter, limit

In [ ]:
# Look at the description table
description = tables["description"]
description

In [ ]:
# Select specific columns
description.select("entity_id", "value").limit(10)

In [ ]:
# Filter rows
id_table = tables["id"]
id_table.filter(id_table["alias"].notnull())

## 4. Joins

In [ ]:
# Join id and description by entity_id
joined = id_table.join(
    description,
    id_table.entity_id == description.entity_id,
).select(
    path=id_table.path,
    key=id_table.key,
    description=description.value,
)
joined.limit(20)

## 5. Aggregation and group_by

In [ ]:
# Count entities per component table
for name, tbl in tables.items():
    n = tbl.entity_id.nunique().execute()
    print(f"{name}: {n} entities")

In [ ]:
# Group by with aggregation — e.g., entities with multiple descriptions
description.group_by("entity_id").aggregate(
    n_descriptions=description.value.count()
).filter(ibis._.n_descriptions > 1)

## 6. Mutate — add computed columns

In [ ]:
# Add a column that shows description length
description.mutate(
    desc_length=description.value.length()
).order_by(ibis.desc("desc_length")).limit(10)

## 7. Method chaining

Ibis expressions compose naturally. Use `ibis._` to refer to the
current table in a chain.

In [ ]:
# Chain: join, filter, select, order
(
    id_table
    .join(description, "entity_id")
    .mutate(desc_length=description.value.length())
    .filter(ibis._.desc_length > 20)
    .select("path", "value", "desc_length")
    .order_by(ibis.desc("desc_length"))
    .limit(10)
)

## 8. SQL escape hatch

You can always drop down to raw SQL when needed.

In [ ]:
con.sql("SELECT * FROM description LIMIT 5")

## 9. Convert back to pandas

In [ ]:
# Any Ibis expression can be converted to a pandas DataFrame
result_df = description.select("entity_id", "value").to_pandas()
result_df.head()

## 10. Inspect the compiled SQL

Use `ibis.to_sql()` to see what SQL Ibis generates.

In [ ]:
expr = (
    id_table
    .join(description, "entity_id")
    .select("path", "value")
    .limit(10)
)
print(ibis.to_sql(expr))

## 11. Try your own queries

Available tables: `tables.keys()` shows all registered component tables.
Use `.execute()` or `.to_pandas()` to materialize results.